# ICoT Exam - Code Solutions

This notebook contains solutions to the code questions in the ICoT multiplication exam.

**Note:** This is the instructor's solution guide. Students should work with `exam_documentation.ipynb`.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import torch

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## CQ1: Fourier Basis R² Computation

**Student-facing stub (identical to exam_documentation.ipynb)**

In [ ]:
# STUB (student version)
import numpy as np

def construct_fourier_basis():
    n = np.arange(10)
    F = np.zeros((10, 6))
    # TODO: Fill in the basis matrix
    return F

def compute_r_squared(vectors, basis):
    # TODO: Implement R² computation
    pass

def generate_fourier_embeddings(d=8):
    # TODO: Create embeddings with Fourier structure
    pass

### SOLUTION

In [ ]:
# SOLUTION - CQ1
import numpy as np

def construct_fourier_basis():
    """Construct Fourier basis matrix F (10×6)"""
    n = np.arange(10)
    F = np.zeros((10, 6))
    
    # Column 0: Constant
    F[:, 0] = 1
    
    # Columns 1-2: k=1 (frequency 2π/10)
    F[:, 1] = np.cos(2 * np.pi * n / 10)
    F[:, 2] = np.sin(2 * np.pi * n / 10)
    
    # Columns 3-4: k=2 (frequency 2π/5)
    F[:, 3] = np.cos(2 * np.pi * n / 5)
    F[:, 4] = np.sin(2 * np.pi * n / 5)
    
    # Column 5: Parity (-1)^n
    F[:, 5] = (-1) ** n
    
    return F

def compute_r_squared(vectors, basis):
    """
    Compute R² for each vector's fit to the basis.
    
    Args:
        vectors: (10, d) array
        basis: (10, k) Fourier basis matrix
    Returns:
        r_squared: (d,) array of R² values
    """
    n_vectors = vectors.shape[1]
    r_squared = np.zeros(n_vectors)
    
    for i in range(n_vectors):
        x = vectors[:, i]
        
        # Fit coefficients using least squares: C = (F^T F)^(-1) F^T x
        C = np.linalg.lstsq(basis, x, rcond=None)[0]
        
        # Predictions
        x_pred = basis @ C
        
        # R² = 1 - SS_res / SS_tot
        ss_res = np.sum((x - x_pred) ** 2)
        ss_tot = np.sum((x - np.mean(x)) ** 2)
        
        r_squared[i] = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    return r_squared

def generate_fourier_embeddings(d=8, noise_scale=0.1):
    """
    Generate embeddings with Fourier structure.
    
    Args:
        d: embedding dimension
        noise_scale: amount of noise to add
    Returns:
        embeddings: (10, d) matrix
    """
    F = construct_fourier_basis()  # (10, 6)
    
    # Random coefficients for combining Fourier basis
    coefficients = np.random.randn(6, d)
    
    # Embeddings as linear combination of Fourier basis
    embeddings = F @ coefficients
    
    # Add small noise
    embeddings += noise_scale * np.random.randn(10, d)
    
    return embeddings

# Execute
F = construct_fourier_basis()
print("Fourier basis shape:", F.shape)
print("Fourier basis (first 3 digits):")
print(F[:3, :])
print()

embeddings = generate_fourier_embeddings(d=8, noise_scale=0.05)
r_squared_values = compute_r_squared(embeddings, F)

median_r2 = np.median(r_squared_values)
print(f"R² values: {r_squared_values}")
print(f"\nMedian R²: {median_r2:.4f}")
print(f"Min R²: {np.min(r_squared_values):.4f}")
print(f"Max R²: {np.max(r_squared_values):.4f}")

### Auto-Check CQ1

In [ ]:
# Auto-check CQ1
assert median_r2 > 0.80, f"Expected median R² > 0.80, got {median_r2:.4f}"
assert F.shape == (10, 6), f"Fourier basis should be (10, 6), got {F.shape}"
print("✓ CQ1 checks passed!")

## CQ2: Long-Range Dependency Pattern

**Student-facing stub**

In [ ]:
# STUB (student version)
def get_dependency_strength(i, j, k):
    # TODO: Implement
    pass

def analyze_dependencies_for_digit(k, max_digit_index=3):
    # TODO: Implement
    pass

### SOLUTION

In [ ]:
# SOLUTION - CQ2
import numpy as np
import matplotlib.pyplot as plt

def get_dependency_strength(i, j, k):
    """
    Compute contribution strength of (ai, bj) to output digit ck.
    
    Logic:
    - i+j = k: Direct contribution (maximum strength)
    - i+j < k: Carry contribution (decreasing with distance)
    - i+j > k: No contribution
    """
    if i + j > k:
        return 0.0  # No contribution
    elif i + j == k:
        return 1.0  # Direct contribution (maximum)
    else:
        # Carry contribution: decreases with distance from k
        distance = k - (i + j)
        # Exponential decay: farther carries have less influence
        return 0.5 ** distance

def analyze_dependencies_for_digit(k, max_digit_index=3):
    """
    Create dependency matrix showing which (i,j) pairs affect ck.
    
    Returns:
        matrix: (4, 4) array of contribution strengths
    """
    matrix = np.zeros((max_digit_index + 1, max_digit_index + 1))
    
    for i in range(max_digit_index + 1):
        for j in range(max_digit_index + 1):
            matrix[i, j] = get_dependency_strength(i, j, k)
    
    return matrix

# Analyze for k=3
k = 3
dep_matrix = analyze_dependencies_for_digit(k)

print(f"Dependency analysis for output digit c{k}:\n")
print("Dependency matrix (rows=ai index, cols=bj index):")
print(dep_matrix)
print()

# Identify strongest contributors
max_strength = np.max(dep_matrix)
strongest_pairs = np.argwhere(dep_matrix == max_strength)

print(f"Pairs with maximum contribution (strength={max_strength}):")
for pair in strongest_pairs:
    i, j = pair
    print(f"  (a{i}, b{j}) where {i}+{j}={i+j}")

print(f"\nExpected pattern: pairs where i+j={k} have maximum strength")
print(f"Confirmed: {all(i+j == k for i, j in strongest_pairs)}")

# Visualize
plt.figure(figsize=(6, 5))
plt.imshow(dep_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(label='Contribution Strength')
plt.xlabel('bj index')
plt.ylabel('ai index')
plt.title(f'Dependency Matrix for c{k}')
plt.xticks(range(4))
plt.yticks(range(4))

# Annotate cells
for i in range(4):
    for j in range(4):
        plt.text(j, i, f'{dep_matrix[i, j]:.2f}', 
                ha='center', va='center', color='black', fontsize=10)

plt.tight_layout()
plt.show()

### Auto-Check CQ2

In [ ]:
# Auto-check CQ2
# Check that pairs where i+j=3 have maximum strength
expected_max_pairs = [(0,3), (1,2), (2,1), (3,0)]
for i, j in expected_max_pairs:
    assert dep_matrix[i, j] == 1.0, f"Expected max strength at ({i},{j})"

# Check that i+j > 3 have zero strength
assert dep_matrix[2, 2] == 0.0, "Expected zero strength when i+j > k"
assert dep_matrix[3, 1] < 1.0 and dep_matrix[3, 1] > 0, "Expected carry contribution when i+j < k"

print("✓ CQ2 checks passed!")

## CQ3: Learning Dynamics Simulation

**Student-facing stub**

In [ ]:
# STUB (student version)
def compute_difficulty_score(digit_idx):
    # TODO: Implement
    pass

def simulate_sft_learning(num_steps=100):
    # TODO: Implement
    pass

def simulate_icot_learning(num_steps=100):
    # TODO: Implement
    pass

### SOLUTION

In [ ]:
# SOLUTION - CQ3
import numpy as np
import matplotlib.pyplot as plt

def compute_difficulty_score(digit_idx):
    """
    Difficulty based on long-range dependencies.
    
    Pattern:
    - c0, c1: Easy (local patterns)
    - c3-c6: Hard (many long-range dependencies)
    - c7: Medium (last digit, some shortcuts)
    """
    # Difficulty increases toward middle, then decreases
    # Parabolic shape centered around middle digits
    if digit_idx in [0, 1]:
        return 0.2  # Easy
    elif digit_idx == 7:
        return 0.4  # Medium
    else:  # 2, 3, 4, 5, 6
        # Difficulty peaks at middle digits
        center_distance = abs(digit_idx - 4.5)
        return 0.8 - 0.1 * center_distance

def simulate_sft_learning(num_steps=100):
    """
    SFT: Edge digits learn, middle plateaus.
    
    Mechanism:
    - Easy digits (c0, c1, c7) learn with fast exponential decay
    - Hard digits (c3-c6) improve initially but plateau
    - Gradient flow cutoff: after easy digits learn, they stop contributing gradients
    """
    losses = np.zeros((8, num_steps))
    
    for digit_idx in range(8):
        difficulty = compute_difficulty_score(digit_idx)
        initial_loss = 1.0
        
        if digit_idx in [0, 1, 7]:  # Edge digits
            # Fast exponential decay to near-zero
            learning_rate = 0.08 if digit_idx in [0, 1] else 0.05
            final_loss = 0.1
            for t in range(num_steps):
                losses[digit_idx, t] = final_loss + (initial_loss - final_loss) * np.exp(-learning_rate * t)
        else:  # Middle digits
            # Initial improvement, then plateau
            plateau_loss = 0.5 + difficulty * 0.4
            improvement_phase = 30
            
            for t in range(num_steps):
                if t < improvement_phase:
                    # Gradual improvement
                    progress = t / improvement_phase
                    losses[digit_idx, t] = initial_loss - (initial_loss - plateau_loss) * progress
                else:
                    # Plateau with small random fluctuation
                    losses[digit_idx, t] = plateau_loss + 0.02 * np.sin(t / 5)
    
    return losses

def simulate_icot_learning(num_steps=100):
    """
    ICoT: All digits learn successfully.
    
    Mechanism:
    - Progressive supervision maintains gradient flow
    - All digits eventually reach low loss
    - Hard digits take longer but don't plateau
    """
    losses = np.zeros((8, num_steps))
    
    for digit_idx in range(8):
        difficulty = compute_difficulty_score(digit_idx)
        initial_loss = 1.0
        final_loss = 0.05
        
        # Learning rate inversely proportional to difficulty
        learning_rate = 0.06 / (1 + difficulty)
        
        for t in range(num_steps):
            losses[digit_idx, t] = final_loss + (initial_loss - final_loss) * np.exp(-learning_rate * t)
    
    return losses

# Run simulations
print("Difficulty scores:")
for i in range(8):
    print(f"  c{i}: {compute_difficulty_score(i):.2f}")
print()

sft_losses = simulate_sft_learning(num_steps=100)
icot_losses = simulate_icot_learning(num_steps=100)

print("Final losses (step 100):")
print(f"SFT:  {sft_losses[:, -1]}")
print(f"ICoT: {icot_losses[:, -1]}")
print()

# Identify plateauing digits in SFT
sft_plateau_digits = [i for i in range(8) if sft_losses[i, -1] > 0.5]
print(f"SFT digits that plateau (final loss > 0.5): {sft_plateau_digits}")
print(f"Expected: [2, 3, 4, 5, 6] or subset")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# SFT learning curves
for i in range(8):
    linestyle = '-' if i in [0, 1, 7] else '--'
    linewidth = 2 if i in [0, 1, 7] else 1.5
    ax1.plot(sft_losses[i, :], label=f'c{i}', linestyle=linestyle, linewidth=linewidth)

ax1.set_xlabel('Training Step')
ax1.set_ylabel('Loss')
ax1.set_title('SFT Learning Dynamics')
ax1.legend(loc='right', ncol=2, fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0.5, color='r', linestyle=':', alpha=0.5, label='Plateau threshold')

# ICoT learning curves
for i in range(8):
    ax2.plot(icot_losses[i, :], label=f'c{i}', linewidth=1.5)

ax2.set_xlabel('Training Step')
ax2.set_ylabel('Loss')
ax2.set_title('ICoT Learning Dynamics')
ax2.legend(loc='right', ncol=2, fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Auto-Check CQ3

In [ ]:
# Auto-check CQ3
# Check SFT edge digits reach low loss
assert sft_losses[0, -1] < 0.2, f"c0 should learn in SFT, got {sft_losses[0, -1]:.3f}"
assert sft_losses[1, -1] < 0.2, f"c1 should learn in SFT, got {sft_losses[1, -1]:.3f}"
assert sft_losses[7, -1] < 0.5, f"c7 should partially learn in SFT, got {sft_losses[7, -1]:.3f}"

# Check SFT middle digits plateau
middle_digit_plateaus = sum(1 for i in [3, 4, 5] if sft_losses[i, -1] > 0.5)
assert middle_digit_plateaus >= 2, f"At least 2 middle digits should plateau in SFT"

# Check ICoT all digits reach low loss
icot_all_low = all(icot_losses[i, -1] < 0.2 for i in range(8))
assert icot_all_low, "All digits should reach low loss in ICoT"

print("✓ CQ3 checks passed!")